# Expanded moderator audit — frozen policies

Evidence review for the 18 November 2026 audit. This notebook reads the saved evaluation outputs; the evaluation scripts perform inference. Run `evaluate_expanded_audit.py` and `run_workflow_checks.py` from the project root to reproduce them.

The new cases were authored and labeled by the project assistant before scoring. They are not independently human-labeled natural conversations. The 22,921-comment test is the existing WikiDetox holdout, re-evaluated rather than a new test set. Manually perturbed variants are paired stress probes, not independent samples. No policies were tuned during this audit.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'results/expanded_audit').is_dir() else Path.cwd().parent
AUDIT = ROOT / 'results/expanded_audit'
def read(name):
    return json.loads((AUDIT / name).read_text())
summary = read('summary.json')
display(pd.DataFrame.from_dict(read('manifest.json')['packages'], orient='index', columns=['Version']))
print('Model, policy, inference and diagnostic hashes unchanged:', summary['frozen_files_unchanged'])


,Version
scikit-learn,1.6.1
numpy,2.4.6
sentence-transformers,6.3.0
torch,2.15.0


Model, policy, inference and diagnostic hashes unchanged: True


## Personal attacks: existing heldout data and authored stress cases

Precision answers how often an alert is correct; recall answers how many attacks are found. Accuracy is sensitive to the class mix. These two datasets have different distributions and should not be averaged.


In [2]:
attack = read('attack_stress_metrics.json')
display(pd.DataFrame(attack['models']).T[['n','accuracy','attack_precision','attack_recall','macro_f1','fp','fn']])
print('Ambiguous cases excluded:', attack['excluded_ambiguous'])
display(pd.read_csv(AUDIT / 'attack_stress_categories.csv')[['category','n','attack_precision','attack_recall','fp','fn']])
display(pd.read_csv(AUDIT / 'existing_holdout_slices.csv')[['slice','n','attack_precision','attack_recall','fp','fn']])


,n,accuracy,attack_precision,attack_recall,macro_f1,fp,fn
alert_prediction,168,0.803571,0.822785,0.77381,0.803397,14,19
LinearSVC,168,0.779762,0.821918,0.714286,0.778814,13,24
LogisticRegression,168,0.809524,0.795455,0.833333,0.809416,18,14


Ambiguous cases excluded: 12


,category,n,attack_precision,attack_recall,fp,fn
0,boundaries_and_format,12,0.666667,1.000000,3,0
1,claim_person_contrasts,12,0.833333,0.833333,1,1
2,explicit_insult,12,1.000000,0.833333,0,2
3,impersonal_profanity,12,0.750000,1.000000,2,0
4,long_mixed_turn,12,1.000000,0.500000,0,3
5,negation,12,0.500000,0.666667,4,2
6,obfuscation_typographical,12,1.000000,0.833333,0,1
7,person_not_participant,12,1.000000,1.000000,0,0
8,polite_surface,12,1.000000,0.833333,0,1
9,punctuation_free_disfluency,12,1.000000,0.833333,0,1


,slice,n,attack_precision,attack_recall,fp,fn
0,all_existing_test,22921,0.803723,0.741321,485,693
1,length_1_to_10,3553,0.833603,0.847291,103,93
2,length_11_to_40,9033,0.820320,0.753673,191,285
3,length_41_to_100,6241,0.776639,0.668430,109,188
4,length_over_100,4094,0.727575,0.632948,82,127
5,contains_second_person,11400,0.804989,0.756526,344,457
6,contains_negation,11578,0.761647,0.661336,220,360
7,contains_quote_mark,138,1.000000,0.285714,0,5
8,contains_profanity_token,1140,0.912432,0.923414,81,70
9,contains_question_mark,6075,0.763727,0.685075,142,211


## Robustness and actual mistakes

Formatting checks and simulated transcript noise use variants of the same original cases. They are not genuine ASR recordings. The ambiguous-context cases remain visible in the row-level CSV but do not enter primary scores.


In [3]:
display(pd.DataFrame(read('attack_perturbation_metrics.json'))[['transform','n','changed_decisions','accuracy','attack_recall']])
errors = pd.read_csv(AUDIT / 'attack_stress_errors.csv')
with pd.option_context('display.max_colwidth', 120):
    display(errors[['category','text','attack','alert_prediction','attack_probability']])


,transform,n,changed_decisions,accuracy,attack_recall
0,chat_prefix,168,8,0.767857,0.690476
1,disfluency_prefix,168,9,0.761905,0.678571
2,extra_whitespace,168,0,0.803571,0.773810
3,html_wrap,168,0,0.803571,0.773810
4,punctuation_removed,168,1,0.809524,0.785714
5,uppercase,168,0,0.803571,0.773810


,category,text,attack,alert_prediction,attack_probability
0,explicit_insult,You have absolutely no brain.,1,0,0.331645
1,explicit_insult,Nobody needs to hear your worthless opinions.,1,0,0.087541
2,claim_person_contrasts,That is a stupid argument because it assumes unlimited money.,0,1,0.770909
3,claim_person_contrasts,You are irrelevant and nobody cares what you think.,1,0,0.046526
4,negation,You are not stupid; the instructions were unclear.,0,1,0.930922
5,negation,I do not think you are an idiot.,0,1,0.998410
6,negation,You are not a liar simply because we disagree.,0,1,0.696119
7,negation,Nobody here is calling you a fool.,0,1,0.721730
8,negation,I am not going to pretend you have a brain.,1,0,0.180255
9,negation,You are not worth listening to.,1,0,0.067522


## Topic monitoring: complete dialogue episodes

Two off-topic turns, optionally separated by acknowledgements, form an episode. A correct alert must occur inside that episode; duplicates or alerts after the return are penalized. Turn classification is shown separately so event suppression does not hide individual relevance errors.


In [4]:
topic = read('topic_stress_metrics.json')
display(pd.Series({k: topic[k] for k in ['sessions','turns','events','detected','false_alerts','event_precision','event_recall','event_f1','median_delay_turns','negative_control_sessions','negative_controls_without_alerts']}))
display(pd.Series(topic['clear_turn_classification']))
display(pd.read_csv(AUDIT / 'topic_stress_sessions.csv'))
turns = pd.read_csv(AUDIT / 'topic_stress_turns.csv')
with pd.option_context('display.max_colwidth', 140):
    display(turns[(turns['alert']) & (turns['reference'] != 'off_topic')][['session','turn','text','reference','state','score']])


sessions                             24.000000
turns                               288.000000
events                               18.000000
detected                             17.000000
false_alerts                          2.000000
event_precision                       0.894737
event_recall                          0.944444
event_f1                              0.918919
median_delay_turns                    1.000000
negative_control_sessions             8.000000
negative_controls_without_alerts      7.000000
dtype: float64

n                                       271
positive_references                      51
accuracy                           0.915129
macro_f1                           0.874651
tn                                      201
fp                                       19
fn                                        4
tp                                       47
accuracy_wilson95      [0.875872, 0.942782]
precision_wilson95     [0.593628, 0.807281]
recall_wilson95        [0.815001, 0.969078]
off_topic_precision                0.712121
off_topic_recall                   0.921569
off_topic_f1                       0.803419
positive_class                    off_topic
dtype: object

,session,category,event_precision,event_recall,event_f1,false_alerts,events,detected,median_delay_turns,false_alerts_per_session
0,topic_stress_01,negative_opposing_stance,0.0,0.0,0.0,0,0,0,NaN,0.0
1,topic_stress_02,negative_semantic_paraphrase,0.0,0.0,0.0,0,0,0,NaN,0.0
2,topic_stress_03,negative_followups,0.0,0.0,0.0,0,0,0,NaN,0.0
3,topic_stress_04,negative_related_examples,0.0,0.0,0.0,0,0,0,NaN,0.0
4,topic_stress_05,negative_procedure,0.0,0.0,0.0,0,0,0,NaN,0.0
5,topic_stress_06,negative_critical_language,0.0,0.0,0.0,0,0,0,NaN,0.0
6,topic_stress_07,negative_long_turns,0.0,0.0,0.0,0,0,0,NaN,0.0
7,topic_stress_08,negative_transcript_like,0.0,0.0,0.0,1,0,0,NaN,1.0
8,topic_stress_09,sudden_drift,1.0,1.0,1.0,0,1,1,1.0,0.0
9,topic_stress_10,sudden_drift,1.0,1.0,1.0,0,1,1,1.0,0.0


,session,turn,text,reference,state,score
93,topic_stress_08,10,an internal review could check whether the same rule is applied fairly,on_topic,off_topic,0.124425
128,topic_stress_11,9,An independent auditor could inspect deletion logs.,on_topic,off_topic,0.029294


## Workflow integrity and warm local text timing

Unit tests use deterministic doubles to isolate contracts. The production wrapper was also exercised on every dialogue turn with the saved models. Long/repeated and concurrent runs test state/performance, not linguistic quality. Timing excludes audio, ASR, browser rendering, network and queueing.


In [5]:
display(pd.Series(read('workflow_unit_checks.json')))
display(pd.Series(read('production_wrapper_checks.json')))
display(pd.Series(read('workload_checks.json')))
assert read('workflow_unit_checks.json')['passed']
assert summary['frozen_files_unchanged']
assert read('existing_holdout_metrics.json')['all_predictions_reproduced']


tests_run                                                   27
failures                                                    []
errors                                                      []
skipped                                                     []
passed                                                    True
scope        Deterministic test doubles validate session lo...
dtype: object

matched_topic_predictions                                                     288
speaker_attribution_verified                                                 True
prior_states_unchanged                                                       True
sessions                                                                       24
warm_median_ms                                                          13.884979
warm_p95_ms                                                             19.787842
timing_scope                    Warm text wrapper on this local CPU, excludes ...
dtype: object

long_session_turns                                                               250
long_session_alerts                                                               83
warm_end_to_end_median_ms                                                  13.672041
warm_end_to_end_p95_ms                                                     15.815885
first_50_median_ms                                                         13.563354
last_50_median_ms                                                          14.684917
concurrent_independent_sessions                                                    4
parallel_workers                                                                   2
concurrent_session_turns                                                          20
cross_session_contamination                                                    False
scope                              Synthetic repetition is a state/performance wo...
dtype: object

## Interpretation and next validation

Strong original-domain accuracy does not imply dependable moderation on spoken debate. Negation, reported abuse, implicit degradation, long turns and contextual follow-ups need targeted improvement. A larger authored diagnostic reveals errors but does not replace independent annotation or natural recordings. Keep these cases as regression tests; after using them for improvement, evaluate the next version on a new locked test set.

See `docs/evaluation/2026-11-18-expanded-audit-report.md` for the Arabic findings and reproducibility commands.
